In [1]:
from ultralytics import YOLO
import time
import cv2
import numpy as np
import onnxruntime as ort
import torch

In [1]:
model = YOLO('onnx/helmet_detection_epoch30/best_helmet_detection_epoch30.pt')

# opset=12 이상 권장. simplify=True로 ONNX 연산 그래프 최적화
save_dir = 'onnx/helmet_detection_epoch30'
output_path = model.export(format='onnx', opset=12, simplify=True, project=save_dir, name='v1')

print(f'ONNX 변환 완료: {output_path}')

Ultralytics 8.4.87 🚀 Python-3.11.15 torch-2.12.0 CPU (Apple M4)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'onnx/helmet_detection_epoch30/best_helmet_detection_epoch30.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (5.9 MB)

ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 1.1s, saved as 'onnx/helmet_detection_epoch30/best_helmet_detection_epoch30.onnx' (11.7 MB)

Export complete (1.2s)
Results saved to /Users/jeongjaehun/Github/03_object_detection/onnx/helmet_detection_epoch30/best_helmet_detection_epoch30.onnx
Predict:         yolo predict task=detect model=onnx/helmet_detection_epoch30/best_helmet_detection_epoch30.onnx imgsz=640 
Validate:        yolo val task=detect model=onnx/helmet_detection_epoch30/best_helmet_detection_epoch30.onnx imgsz=640 data=/content/working/HelmetDataset/data.yaml  
Visualize:       https://netron.app
O

In [8]:
onnx_path = 'onnx/helmet_detection_epoch30/best_helmet_detection_epoch30.onnx'

# 1. ONNX Runtime 세션 생성
# M4 맥미니 환경에서는 CPUProvider 사용
session = ort.InferenceSession(onnx_path, providers=['CoreMLExecutionProvider'])

# 입력/출력 노드 이름 확인
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

for i in range(5):
    # 2. 테스트 이미지 로드 및 전처리  (YOLOv8 입력 규격: 640 640, RGB, Normalize)
    image_path = 'onnx/helmet_detection_epoch30/BikesHelmets0.png'
    img = cv2.imread(image_path)
    h_orig, w_orig, _ = img.shape
    
    # Resize & BGR -> RGB
    img_resized = cv2.resize(img, (640, 640))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    # HWC -> CHW, 0~1 Normalize, Batch 차원 추가 (1, 3, 640, 640)
    input_tensor = img_rgb.transpose(2, 0, 1).astype(np.float32) / 255.0
    input_tensor = np.expand_dims(input_tensor, axis=0)
    
    # 3. ONNX Runtime 추론 및 Latency 측정
    start_time = time.time()
    outputs = session.run([output_name], {input_name: input_tensor})
    latency = (time.time() - start_time) * 1000 # ms 단위
    
    print(f'ONNX Runtime 추론 소요 시간: {latency:.2f} ms')
    print(f'출력 텐서 Shape: {outputs[0].shape}\n')

ONNX Runtime 추론 소요 시간: 8.55 ms
출력 텐서 Shape: (1, 6, 8400)

ONNX Runtime 추론 소요 시간: 7.65 ms
출력 텐서 Shape: (1, 6, 8400)

ONNX Runtime 추론 소요 시간: 8.53 ms
출력 텐서 Shape: (1, 6, 8400)

ONNX Runtime 추론 소요 시간: 7.38 ms
출력 텐서 Shape: (1, 6, 8400)

ONNX Runtime 추론 소요 시간: 5.95 ms
출력 텐서 Shape: (1, 6, 8400)



2026-09-03 17:26:30.724 python3[1416:15512] 2026-09-03 17:26:30.797686 [W:onnxruntime:, coreml_execution_provider.cc:137 GetCapability] CoreMLExecutionProvider::GetCapability, number of partitions supported by CoreML: 3 number of nodes in the graph: 232 number of nodes supported by CoreML: 228
2026-09-03 17:26:31.446 python3[1416:15512] 2026-09-03 17:26:31.519816 [W:onnxruntime:, session_state.cc:1387 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2026-09-03 17:26:31.446 python3[1416:15512] 2026-09-03 17:26:31.519836 [W:onnxruntime:, session_state.cc:1389 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.


In [45]:
pt_path = 'onnx/helmet_detection_epoch30/best_helmet_detection_epoch30.pt'

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)

model = YOLO(pt_path).to(device)



# Warm up (첫 번째 추론은 GPU/CPU 메모리 할당으로 인해 느릴 수 있으므로 1회 미리 실행)
_ = model(image_path, verbose=False)
for i in range(5):
    results = model(image_path)
    
    speed_dict = results[0].speed
    print(f"전처리 시간: {speed_dict['preprocess']:.2f} ms")
    print(f"순수 추론 시간: {speed_dict['inference']:.2f} ms")
    print(f"후처리 시간: {speed_dict['postprocess']:.2f} ms")
    
    total_time = sum(speed_dict.values())
    print(f'총 소요 시간: {total_time:.2f} ms\n')

mps
전처리 시간: 2.01 ms
순수 추론 시간: 9.29 ms
후처리 시간: 5.15 ms
총 소요 시간: 16.45 ms

전처리 시간: 2.46 ms
순수 추론 시간: 10.73 ms
후처리 시간: 2.89 ms
총 소요 시간: 16.07 ms

전처리 시간: 1.72 ms
순수 추론 시간: 6.70 ms
후처리 시간: 3.42 ms
총 소요 시간: 11.84 ms

전처리 시간: 1.43 ms
순수 추론 시간: 6.35 ms
후처리 시간: 3.37 ms
총 소요 시간: 11.15 ms

전처리 시간: 1.73 ms
순수 추론 시간: 6.19 ms
후처리 시간: 4.28 ms
총 소요 시간: 12.20 ms

